# Pipeline Flow Debug (LangGraph)

This notebook runs the **full agentic pipeline** via **LangGraph** (no FastAPI, no Chainlit).

Prereqs:
- Put keys in `.env.local`: `OPENAI_API_KEY` and `TAVILY_API_KEY` (and/or `ANTHROPIC_API_KEY` depending on `config/agents.yaml`).
- Install deps: `pip install -r requirements.txt`

Notes:
- Each LangGraph node instantiates a fresh agent, so edits to `config/agents.yaml` / `config/prompts.yaml` take effect on the next run without restarting the kernel.
- Detailed I/O logs go to `logs/agent_io.jsonl` (filter by `run_id`).


In [ ]:
import sys
from pathlib import Path
import json
from datetime import datetime

# Find repo root (directory containing both `src/` and `config/`).
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not ((repo_root / 'src').exists() and (repo_root / 'config').exists()):
    repo_root = repo_root.parent

if not ((repo_root / 'src').exists() and (repo_root / 'config').exists()):
    raise RuntimeError(f'Could not locate repo root from cwd={Path.cwd()}')

# Ensure repo root is on sys.path so `import src.*` works reliably.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.env import load_env
load_env()

from src.state.schema import PipelineState
from src.graph.builder import build_pipeline_graph


In [ ]:
def state_summary(state: dict) -> dict:
    def _n(key: str) -> int:
        v = state.get(key)
        return len(v) if isinstance(v, list) else 0

    return {
        'run_id': state.get('run_id'),
        'topic': state.get('topic'),
        'current_agent': state.get('current_agent'),
        'pipeline_status': state.get('pipeline_status'),
        'quality_iteration': state.get('quality_iteration'),
        'quality_verdict': state.get('quality_verdict'),
        'quality_score': state.get('quality_score'),
        'counts': {
            'research_queries': _n('research_queries'),
            'raw_sources': _n('raw_sources'),
            'deduplicated_sources': _n('deduplicated_sources'),
            'clinical_claims': _n('clinical_claims'),
            'evidence_gaps': _n('evidence_gaps'),
            'citations': _n('citations'),
            'quality_issues': _n('quality_issues'),
        },
        'report_word_count': state.get('report_word_count'),
        'error_message': state.get('error_message'),
    }


def build_initial_state(
    *,
    topic: str,
    scope_instructions: str = '',
    target_audience: str = 'clinical practitioners',
    report_format: str = 'clinical_brief',
    max_quality_iterations: int = 3,
) -> PipelineState:
    return PipelineState(
        run_id='notebook-' + datetime.now().strftime('%Y%m%d-%H%M%S'),
        topic=topic,
        scope_instructions=scope_instructions,
        target_audience=target_audience,
        report_format=report_format,
        requested_at=datetime.now(),
        research_queries=[],
        scope_boundaries={'in_scope': [], 'out_of_scope': []},
        priority_subtopics=[],
        raw_sources=[],
        deduplicated_sources=[],
        research_summary='',
        clinical_claims=[],
        evidence_gaps=[],
        contradictions=[],
        statistical_findings=[],
        analysis_narrative='',
        report_sections={},
        report_markdown='',
        citations=[],
        report_word_count=0,
        quality_issues=[],
        quality_verdict='pending',
        quality_score=0.0,
        revision_instructions='',
        quality_iteration=0,
        max_quality_iterations=max_quality_iterations,
        should_revise=False,
        current_agent='',
        agent_history=[],
        pipeline_status='pending',
        error_message=None,
    )


In [ ]:
# Edit these, then run the next cell(s).
TOPIC = 'Effectiveness of telemedicine interventions for Type 2 diabetes management in adults'
SCOPE = 'Focus on adults 18+, exclude pediatric cases. Prefer evidence from 2018-2026.'
AUDIENCE = 'clinical practitioners'
REPORT_FORMAT = 'clinical_brief'  # or 'full_report'
MAX_QUALITY_ITERATIONS = 3

state = build_initial_state(
    topic=TOPIC,
    scope_instructions=SCOPE,
    target_audience=AUDIENCE,
    report_format=REPORT_FORMAT,
    max_quality_iterations=MAX_QUALITY_ITERATIONS,
)

state_summary(state)


## Run Full Pipeline

This calls `build_pipeline_graph()` and runs the compiled graph with `await graph.ainvoke(state)`.
If the quality verdict is `revise`, the graph may loop `quality -> writing` until `max_quality_iterations` is reached.


In [ ]:
graph = build_pipeline_graph()

t0 = datetime.now()
result = await graph.ainvoke(state)
dt = (datetime.now() - t0).total_seconds()

state = result
print(f"Pipeline finished in {dt:.1f}s | status={state.get('pipeline_status')} | verdict={state.get('quality_verdict')}")
print(json.dumps(state_summary(state), indent=2))

print('\nReport preview:\n')
report = (state.get('report_markdown') or '')
print(report[:2500])


## Inspect Latest Logs For This Run

This greps `logs/agent_io.jsonl` for the current `run_id`.


In [ ]:
from pathlib import Path

log_path = repo_root / 'logs' / 'agent_io.jsonl'
run_id = state.get('run_id')
if not log_path.exists():
    print(f"No log file found at {log_path}")
else:
    lines = log_path.read_text(encoding='utf-8').splitlines()
    hits = [ln for ln in lines if f'\"run_id\": \"{run_id}\"' in ln]
    print(f"Log hits for run_id={run_id}: {len(hits)}")
    for ln in hits[-25:]:
        print(ln[:4000])
